# OpenEars - getting testdata from Sensemakers IoT Platform

## Contents
1. Intro
2. Getting raw data from Sensemakers IoT Platform
3. Cleaning data
4. Show highest class with prediction for last 10 minutes

## 1. Intro

With this Notebook one can get (test) data from the Sensemakers IoT Platform and load it into a Pandas df. 

As of now, you'll need to start the RPi manually and keep the microphone close to a speaker.
Starting the Pi remotely is not (yet) possible.

#### Flowchart
<img src="drawio_openears-flow.png">

#### Credentials
If you want to use this notebook, you'll need a config file with credentials. 
Please contact Sensemakers to receive the credentials.  

#### Links:
- OpenEars GitHub Sensemakers - https://github.com/sensemakersamsterdam/OpenEars
- Sensemakers IoT Platform - https://github.com/sensemakersamsterdam/sensemakers-iot-platform

## 2. Getting raw data from Sensemakers IoT Platform and write to Pandas Dataframe

In [ ]:
#source: https://github.com/sensemakersamsterdam/sensemakers-iot-platform/tree/master/scripts
# -*- coding: utf-8 -*-
"""Read data from InfluxDB into Pandas DataFrame."""

from influxdb import DataFrameClient
import os
import oe_config

influxdb_host = oe_config.influxdb_host
influxdb_port = oe_config.influxdb_port
influxdb_user = oe_config.influxdb_user
influxdb_password = oe_config.influxdb_password
influxdb_dbname = oe_config.influxdb_dbname

client = DataFrameClient(
    influxdb_host,
    influxdb_port,
    influxdb_user,
    influxdb_password,
    influxdb_dbname,
    ssl=True,
    verify_ssl=True,
)

# Query returns a dictionary of dataframes.
rs = client.query(
    "SELECT * FROM /.*/ WHERE app_id='openears' AND dev_id='OE001'") #change the dev_id for your own device

#print(rs.keys())
# Specify the data you want to retrieve.
df = rs["openears/OE001"] #change the dev_id for the right device 
df.tail(5) #print the last 5 recordings. See below. Time = UTC

556 gunshots                     5
Brommer                          2
Luide mensen (schreeuwen enz)    3
Motoren                          5
Terrasgeluide (stille mensen)    5
Versterkte muziek                4
app_id                           6
cputemp                          6
dev_id                           6
dtype: int64

## 3. Data cleaning 

In [2]:
#first we will drop all old classes etc.
df.columns # show all columns

Index(['556 gunshots', 'Accelerating, revving, vroom', 'Aircraft', 'Brommer',
       'Brommer Alarm', 'Car', 'Car passing by', 'Claxons',
       'Dichtklappende autodeuren', 'Dog', 'Emergency vehicle',
       'Gunshot, gunfire', 'Jackhammer', 'Luide mensen (schreeuwen enz)',
       'Motor vehicle (road)', 'Motorcycle', 'Motoren', 'Police car (siren)',
       'Silence', 'Terrasgeluide (stille mensen)', 'Tools',
       'Vehicle horn | car horn | honking', 'Versterkte muziek', 'app_id',
       'cpu temp', 'cpu_temp', 'cputemp', 'dev_id', 'humidity', 'temperature',
       'wifi signal'],
      dtype='object')

In [3]:
#drop non-relevant columns and old classes
df = df.drop(['app_id', 'cpu_temp', 'cpu temp','dev_id','humidity', 'temperature', 'wifi signal', 'Accelerating, revving, vroom', 'Aircraft', 'Car', 'Car passing by', 'Dog', 'Emergency vehicle', 'Gunshot, gunfire', 'Jackhammer', 'Motor vehicle (road)', 'Motorcycle', 'Police car (siren)', 'Police car (siren)', 'Tools', 'Vehicle horn | car horn | honking' ], axis=1)
df.columns

Index(['556 gunshots', 'Brommer', 'Brommer Alarm', 'Claxons',
       'Dichtklappende autodeuren', 'Luide mensen (schreeuwen enz)', 'Motoren',
       'Silence', 'Terrasgeluide (stille mensen)', 'Versterkte muziek',
       'cputemp'],
      dtype='object')

In [10]:
df.tail(100)

,556 gunshots,Brommer,Brommer Alarm,Claxons,Dichtklappende autodeuren,Luide mensen (schreeuwen enz),Motoren,Silence,Terrasgeluide (stille mensen),Versterkte muziek,cputemp
2021-09-01 17:01:53.423000+00:00,0.241172,0.121129,NaN,NaN,0.122576,NaN,0.988597,NaN,0.113895,NaN,66.7
2021-09-01 17:02:08.463000+00:00,0.106591,0.127377,NaN,NaN,0.213700,NaN,0.993447,NaN,0.182577,NaN,65.7
2021-09-01 17:02:23.488000+00:00,0.174673,0.138743,NaN,NaN,0.259473,NaN,0.986392,NaN,0.141515,NaN,65.2
2021-09-01 17:02:38.538000+00:00,NaN,0.915198,NaN,NaN,NaN,NaN,0.567320,NaN,NaN,NaN,66.2
2021-09-01 17:02:53.560000+00:00,NaN,0.947654,NaN,0.216968,NaN,NaN,0.465214,NaN,0.109022,NaN,66.7
...,...,...,...,...,...,...,...,...,...,...,...
2021-09-01 17:25:42.394000+00:00,NaN,NaN,NaN,NaN,0.643296,NaN,0.983893,NaN,0.172533,NaN,65.2
2021-09-01 17:25:57.461000+00:00,0.992030,0.254074,NaN,NaN,0.187533,NaN,0.809830,NaN,NaN,NaN,66.2
2021-09-01 17:26:12.476000+00:00,0.259445,0.355284,NaN,NaN,0.403686,NaN,0.773817,NaN,0.108479,NaN,66.2
2021-09-01 17:26:27.537000+00:00,0.165818,NaN,NaN,0.271228,NaN,NaN,0.995925,NaN,NaN,NaN,66.2


In [5]:
# convert last 40 records = 10 minutes to new dataframe
df_10minutes=df.tail(40) 
df_10minutes

,556 gunshots,Brommer,Brommer Alarm,Claxons,Dichtklappende autodeuren,Luide mensen (schreeuwen enz),Motoren,Silence,Terrasgeluide (stille mensen),Versterkte muziek,cputemp
2021-09-01 17:16:55.946000+00:00,NaN,0.995404,NaN,0.255044,0.149768,0.147864,NaN,NaN,0.109819,0.129470,67.2
2021-09-01 17:17:11.001000+00:00,NaN,0.993306,NaN,0.231301,0.146330,0.161494,NaN,NaN,0.117000,0.139336,65.2
2021-09-01 17:17:26.033000+00:00,NaN,0.994436,NaN,0.244317,0.146861,0.151697,0.105008,NaN,0.111180,0.141594,66.2
2021-09-01 17:17:41.066000+00:00,NaN,0.995169,NaN,0.244803,0.147309,0.158041,NaN,NaN,0.108984,0.128449,65.7
2021-09-01 17:17:56.120000+00:00,NaN,0.995200,NaN,0.257314,0.149739,0.160290,NaN,NaN,0.119159,0.129030,66.2
2021-09-01 17:18:11.164000+00:00,NaN,0.994964,NaN,0.253404,0.147003,0.164238,NaN,NaN,0.118474,0.131395,65.7
2021-09-01 17:18:26.206000+00:00,NaN,0.996550,NaN,0.243842,0.151202,0.149580,NaN,NaN,0.106008,0.120856,65.2
2021-09-01 17:18:41.242000+00:00,NaN,0.994976,NaN,0.225356,0.144937,0.146949,0.134250,NaN,0.113512,0.156703,66.2
2021-09-01 17:18:56.295000+00:00,NaN,0.994520,NaN,0.236871,0.137535,0.141389,NaN,NaN,0.115280,0.128075,65.2
2021-09-01 17:19:11.326000+00:00,NaN,0.996262,NaN,0.234170,0.156990,0.148748,NaN,NaN,0.106678,0.145814,64.7


### 3b. Checking the temperature of the Raspberry

In [6]:
average_cpu= df_10minutes['cputemp'].mean()#calculate average temperature of the Pi
max_cpu= df_10minutes['cputemp'].max()#calculate maximum temperature of the Pi
min_cpu= df_10minutes['cputemp'].min()#calculate minimum temperature of the Pi
print(average_cpu, max_cpu, min_cpu)

65.85 67.2 64.2


## 4. Show highest prediction of last 10 minutes

In [7]:
df_10minutes = df_10minutes.drop(['cputemp'], axis=1)

In [8]:
#show highest Class with prediction 
import pandas as pd
result_df = pd.DataFrame(columns = ['Class' , 'Prediction'])
result_df['Class'] = df_10minutes.idxmax(axis=1) # get the name of the column
result_df['Prediction'] = df_10minutes.max(axis=1) # get the highest value per row
result_df #show the results

,Class,Prediction
2021-09-01 17:16:55.946000+00:00,Brommer,0.995404
2021-09-01 17:17:11.001000+00:00,Brommer,0.993306
2021-09-01 17:17:26.033000+00:00,Brommer,0.994436
2021-09-01 17:17:41.066000+00:00,Brommer,0.995169
2021-09-01 17:17:56.120000+00:00,Brommer,0.995200
2021-09-01 17:18:11.164000+00:00,Brommer,0.994964
2021-09-01 17:18:26.206000+00:00,Brommer,0.996550
2021-09-01 17:18:41.242000+00:00,Brommer,0.994976
2021-09-01 17:18:56.295000+00:00,Brommer,0.994520
2021-09-01 17:19:11.326000+00:00,Brommer,0.996262


## 4b. Store the results as a .csv

In [27]:
#CHANGE THE NAME OF THE FILE FORMAT: OEdev_id_YYMMDD_name.csv
df.to_csv('OE001_210720.csv') #change date manually